# Learnability of the charge: mean entropy (Barratt et al. 2022, Fig. 2c)

Issue #115. This notebook reproduces Fig. 2c of F. Barratt, U. Agrawal, A. C. Potter, S. Gopalakrishnan and R. Vasseur, “Transitions in the learnability of global charges from local measurements”, PRL 129, 200602 (2022), and overlays it on the curve digitized from the paper (chain lengths 6 and 20, that is `N = 6` and `N = 20` here). It validates the library functions (`QELM.models.circuits`, `QELM.evaluation.learnability`) on a published result.

The protocol is that of Fig. 2b (`notebooks/benchmarks/01_learnability_accuracy.ipynb`, same script `experiments/learnability/learnability.py`, config `conf/learnability.yaml`):

- a chain of `N` qubits (no ancilla, so `N` is also the chain length, written `L` in the paper) starts from the uniform superposition of the bit strings of charge `Q0 = N // 2` (half of the records) or `Q1 = N // 2 - 1` (the other half);
- it evolves for `t_f = N` steps of a brickwork circuit of random U(1) gates (one step = an even layer then an odd layer, a fresh circuit per record); after each layer, each site is measured in Z with probability `p` (Born rule), that is `2 t_f` measurement rounds (`learnability.measure_every=layer`, the choice that reproduces Fig. 2c, issue #115);
- Eve sees the record but not the gates. Her optimal classifier is the Bayesian posterior `P(Q | m)` computed with the dynamics averaged over the Haar gates, a symmetric exclusion process, and `P_corr = P(Q* | m)` is the posterior of the true charge.

The observable of Fig. 2c is the mean entropy `E[H(P_corr)]` of the binary distribution `{P_corr, 1 - P_corr}`, in nats as in the paper: `ln 2 ≈ 0.693` when Eve knows nothing (`p = 0`), 0 when she knows the charge. It is the order parameter of the success transition; the inset is the Binder ratio of `H(P_corr)`, whose curves for several `N` cross near `p_success ≈ 0.2` in the paper.

The paper does not say at which point of a step the sites are measured. Measuring after each layer reproduces the digitized `N = 6` curve within 0.006 nats (rms 0.003 over 25 points, 6 seeds of 40,000 records; issue #177); measuring once per step (`measure_every=period`) learns the charge about twice too slowly in `p`.

In [ ]:
import os
from pathlib import Path

# Jupyter starts in the notebook's folder. Move to the repository root (the
# folder that contains pyproject.toml) so conf/, experiments/ and outputs/
# resolve.
root = Path.cwd()
while not (root / "pyproject.toml").is_file() and root != root.parent:
    root = root.parent
os.chdir(root)
print(f"cwd: {Path.cwd()}")

## Parameters

Each value can be changed here, or through the environment variable of the same name (this is what the test `tests/test_learnability_entropy_notebook.py` does for a small sweep). Notation (issue #147): `N` is the total number of qubits, here the chain length since there is no ancilla, and `N_VALUES` lists the sizes. The paper writes its chain length `L`: its curves, named `L = 6` in the exports, are shown as `N = 6`, and our curve of the same size is `N6`. `L` is the number of records per point (40,000 in the paper), `samples.L` in the config. Each sweep goes to this notebook's own folder, `outputs/learnability_fig2c/<date_time>/`, and the notebook reads back exactly that folder (issue #151). `RUN = False` reads back the latest sweep of this folder without launching a new one. With several sizes, the Binder cell gives the crossing.

Indicative duration: about 10 s per point at `N = 6` and 1 to 2 min per point at `N = 8`.

In [ ]:
L = os.environ.get("L", "40000")  # records per run (samples.L)
N_VALUES = os.environ.get("N_VALUES", "6")  # total qubits N (no ancilla)
SEEDS = os.environ.get("SEEDS", "0 1 2 3 42 45")
P_VALUES = os.environ.get(
    "P_VALUES",
    "0.0 0.025 0.05 0.075 0.1 0.125 0.15 0.175 0.2 0.25 0.3 0.35 0.4 0.5",
)
OUT_ROOT = Path(os.environ.get("OUT_ROOT", "outputs"))
RUN = os.environ.get("RUN", "1") == "1"

## Launch the sweep

In [ ]:
from QELM.utils import latest_sweep, new_sweep_dir, run_with_progress

# This notebook's own output folder (issue #151): no other notebook writes
# there, so the cells below never read another notebook's sweep.
EXPERIMENT = "learnability_fig2c"

if RUN:
    # Create the sweep folder here and hand it to the launcher (OUT_DIR):
    # the cells below read this sweep and no other.
    sweep_dir = new_sweep_dir(OUT_ROOT, EXPERIMENT)
    env = {
        **os.environ,
        "L": L,
        "N_VALUES": N_VALUES,
        "SEEDS": SEEDS,
        "P_VALUES": P_VALUES,
        "OUT_DIR": str(sweep_dir),
        "FIGURES": "2c",  # only this notebook's figure
    }
    n_runs = len(N_VALUES.split()) * len(SEEDS.split()) * len(P_VALUES.split())
    # One bar for the whole sweep, one step per run; raises
    # CalledProcessError, with the runs' output, if a run fails.
    run_with_progress(
        ["bash", "experiments/learnability/learnability.sh"], total=n_runs, env=env
    )
else:
    # RUN = False: the latest sweep of this notebook only.
    sweep_dir = latest_sweep(OUT_ROOT / EXPERIMENT)
print(f"sweep: {sweep_dir}")

## Read back the metrics

One row per `(N, p)`, averaged over the seeds. The error is the standard error over the seeds, or the bootstrap error over the records with a single seed.

In [ ]:
from QELM.utils import load_sweep, summarize

METRICS = ["entropy_mean", "entropy_var", "entropy_binder", "accuracy"]
rows = load_sweep(
    sweep_dir, keys=(("N", "N"), ("seed", "seed"), ("p", "learnability.p"))
)
table = summarize(
    rows, by=("N", "p"), metrics=[*METRICS, *(f"{m}_err" for m in METRICS)]
)
print(f"{len(rows)} runs")
print(f"{'N':>3} {'p':>6} {'runs':>4} " + " ".join(f"{m:>19}" for m in METRICS))
for r in table:
    errors = [r[f"{m}_sem"] if r["n_runs"] > 1 else r[f"{m}_err_mean"] for m in METRICS]
    values = " ".join(
        f"{r[f'{m}_mean']:>9.4f} ± {e:<7.4f}"
        for m, e in zip(METRICS, errors, strict=True)
    )
    print(f"{r['N']:>3} {r['p']:>6.3f} {r['n_runs']:>4} {values}")

## Overlay on Fig. 2c

On the left the mean entropy, with the curves digitized from the paper dashed (`experiments/learnability/reference/barratt2022_fig2c_entropy.csv`); on the right the Binder ratio of the entropy (the inset of Fig. 2c). The dotted vertical line marks `p_success = 0.2`. `comparison` gives the deviation from the reference of the same `N` (at the reference points, our curve being interpolated) and the `p` where the Binder curves of two sizes cross.

The figure, the data it shows and the comparison are saved in the sweep folder: `learnability_fig2c.png`, `learnability_fig2c.json` and `comparison_fig2c.json`.

In [ ]:
import importlib.util

import matplotlib.pyplot as plt

from QELM.utils.plots import print_comparison

# The plotting script of the experiment is not a package: import it by path.
spec = importlib.util.spec_from_file_location(
    "plot_learnability", "experiments/learnability/plot_learnability.py"
)
plot_learnability = importlib.util.module_from_spec(spec)
spec.loader.exec_module(plot_learnability)

# Figure, its data (.json of the same name) and the comparison file, all in
# the sweep folder. The error bars are smaller than the markers on the scale
# of the entropy: residuals=True adds under the panel our curve minus the
# paper's, where they show.
panels, transition, filename, comparison_name = plot_learnability.FIGURES["2c"]
comparison = plot_learnability.plot_sweep(
    sweep_dir,
    plot_learnability.REFERENCE_DIR,
    panels,
    filename,
    transition=transition,
    comparison_name=comparison_name,
    residuals=True,
)
figure_path = sweep_dir / filename
print_comparison(comparison)
print(f"figure: {figure_path}")

# Draw it again to show it inline (plot_sweep closes its figure).
fig, _ = plot_learnability.draw_sweep(
    sweep_dir, plot_learnability.REFERENCE_DIR, panels, transition, residuals=True
)
plt.show()